# AutoML com AutoGluon usando o dataset Iris

Este notebook apresenta um exemplo didático de **AutoML para classificação** usando o dataset clássico **Iris**.

A proposta é mostrar que, com AutoML, o aluno não escolhe manualmente um único algoritmo. A biblioteca testa diferentes modelos, compara resultados e apresenta um ranking dos melhores modelos.

> **Observação de compatibilidade:** o AutoGluon atualmente suporta Python 3.10, 3.11, 3.12 e 3.13. Não é recomendado para Python 3.14 neste momento.


## 1. Instalação

Execute esta célula apenas se o AutoGluon ainda não estiver instalado no ambiente.

Em Linux com CPU, a instalação recomendada pode ser feita com:

```bash
pip install -U pip setuptools wheel
pip install autogluon --extra-index-url https://download.pytorch.org/whl/cpu
```

Se estiver usando Jupyter, após instalar reinicie o kernel.


In [ ]:
# Execute somente se necessário
# !pip install -U pip setuptools wheel
# !pip install autogluon --extra-index-url https://download.pytorch.org/whl/cpu

## 2. Imports

O AutoGluon trabalha muito bem com dados tabulares em formato `DataFrame`.

Neste exemplo, vamos usar:

- `load_iris` para carregar o dataset;
- `pandas` para organizar os dados;
- `train_test_split` para separar treino e teste;
- `TabularPredictor` para executar o AutoML;
- métricas do Scikit-Learn para avaliação final.


In [ ]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from autogluon.tabular import TabularPredictor

## 3. Carregando o dataset Iris

O dataset Iris contém medidas de flores e uma variável alvo com três classes:

- `setosa`
- `versicolor`
- `virginica`

As variáveis de entrada são medidas de sépala e pétala.


In [ ]:
iris = load_iris()

df = pd.DataFrame(
    data=iris.data,
    columns=iris.feature_names
)

df["target"] = iris.target
df["target_name"] = df["target"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginica"
})

df.head()

## 4. Preparando o dataset para o AutoGluon

Para fins didáticos, vamos usar a coluna textual `target_name` como variável alvo.

A coluna numérica `target` será removida para evitar vazamento de dados, pois ela representa exatamente a resposta.


In [ ]:
label = "target_name"

data = df.drop(columns=["target"])

train_data, test_data = train_test_split(
    data,
    test_size=0.30,
    random_state=42,
    stratify=data[label]
)

print("Treino:", train_data.shape)
print("Teste:", test_data.shape)

train_data.head()

## 5. Treinando o AutoML com AutoGluon

O `TabularPredictor` recebe o nome da coluna alvo.

O método `fit()` inicia o processo de AutoML. Durante esse processo, o AutoGluon pode testar vários modelos, comparar desempenhos e montar ensembles.

Para aula, usamos `time_limit=120`, ou seja, aproximadamente 2 minutos de busca.


In [ ]:
predictor = TabularPredictor(
    label=label,
    problem_type="multiclass",
    eval_metric="accuracy",
    path="autogluon_iris_model"
)

predictor.fit(
    train_data=train_data,
    time_limit=120,
    presets="medium_quality"
)

## 6. Ranking dos modelos treinados

O `leaderboard()` mostra os modelos avaliados pelo AutoGluon.

Ele ajuda o aluno a entender que o AutoML compara várias alternativas e escolhe as melhores conforme a métrica configurada.


In [ ]:
leaderboard = predictor.leaderboard(test_data, silent=True)
leaderboard

## 7. Fazendo previsões

Agora vamos separar as variáveis de entrada e a resposta real do conjunto de teste.

Depois, usaremos o modelo escolhido pelo AutoGluon para prever as classes das flores.


In [ ]:
X_test = test_data.drop(columns=[label])
y_test = test_data[label]

y_pred = predictor.predict(X_test)

pd.DataFrame({
    "real": y_test.values,
    "previsto": y_pred.values
}).head(10)

## 8. Avaliação do modelo

Vamos calcular:

- acurácia;
- relatório de classificação;
- matriz de confusão.

Essas métricas permitem comparar o AutoML com modelos manuais, como árvore de decisão, regressão logística ou KNN.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Acurácia: {accuracy:.4f}")

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

## 9. Probabilidades previstas

Além da classe final, o AutoGluon também pode retornar as probabilidades estimadas para cada classe.


In [ ]:
probas = predictor.predict_proba(X_test)
probas.head()

## 10. Importância das variáveis

A importância das variáveis ajuda a explicar quais atributos mais contribuíram para o modelo.

No caso do Iris, normalmente as medidas das pétalas tendem a ser bastante relevantes.


In [ ]:
feature_importance = predictor.feature_importance(test_data)
feature_importance

## 11. Conclusão didática

Neste exemplo, o AutoGluon executou um fluxo de AutoML para classificação multiclasse.

Em uma abordagem tradicional, o aluno precisaria escolher manualmente um algoritmo, configurar hiperparâmetros, treinar, avaliar e comparar modelos.

Com AutoML, parte desse processo é automatizada:

1. leitura dos dados;
2. identificação do problema;
3. teste de múltiplos modelos;
4. comparação via métrica;
5. escolha do melhor modelo;
6. geração de ranking;
7. predição final.

Mesmo assim, o papel do cientista de dados continua importante para:

- entender o problema;
- preparar os dados corretamente;
- evitar vazamento de dados;
- escolher métricas adequadas;
- interpretar os resultados;
- validar se o modelo faz sentido no contexto real.
